In [13]:
from langchain.agents import create_agent
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnableParallel, branch
from dataclasses import Field

from langchain_core.tools import tool
from  langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx
import urllib3
import ssl

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel
from typing import Literal
import uuid
import base64

from langchain_community.tools import DuckDuckGoSearchRun



# This function will load all the variables from the .env file and will
# make them available in the os.environ dictionary (env variables)
load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

# Disable SSL warnings for urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = llm_openai = ChatOpenAI()
search_tool = DuckDuckGoSearchRun(description="This is a tool to search the web for news")

# Create a custom Wikipedia tool using httpx with SSL configuration
@tool
def wikipedia_search(query: str) -> str:
    """Search Wikipedia for information about a topic. Use this when you need factual information from Wikipedia."""
    try:
        # Use Wikipedia API directly with httpx client
        url = "https://en.wikipedia.org/w/api.php"
        params = {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "format": "json",
            "srlimit": 3
        }
        
        # Create a separate httpx client for Wikipedia with SSL verification disabled
        wiki_client = httpx.Client(verify=False)
        response = wiki_client.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        if data.get("query", {}).get("search"):
            results = data["query"]["search"]
            summary = f"Found {len(results)} Wikipedia results for '{query}':\n\n"
            for i, result in enumerate(results[:3], 1):
                title = result.get("title", "Unknown")
                snippet = result.get("snippet", "").replace('<span class="searchmatch">', "").replace('</span>', "")
                summary += f"{i}. {title}\n   {snippet}\n\n"
            return summary
        else:
            return f"No Wikipedia results found for '{query}'"
    except Exception as e:
        return f"Error searching Wikipedia: {str(e)}"

@tool
def enterprise_tool(query:str)-> str:

    """This is a tool to send emails to employees"""

    return "Email Sent"

toolkit=[search_tool,wikipedia_search,enterprise_tool]
agent=create_agent(model=llm_openai,tools=toolkit)
agent
example_query = "Tell me Paris the capital of France from wiki"

# events = agent.stream(
#     {"messages": [("user", example_query)]},
#     stream_mode="values",
# )
# for event in events:
#     event["messages"][-1].pretty_print()

llm_binded = llm_openai.bind_tools(toolkit)
llm_binded.invoke("What's the latest news about the stock market?")

AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-oss-120b', 'system_fingerprint': 'vllm-0.27.1-tp4-8684504f', 'model_provider': 'openai'}, id='lc_run--01a0e888-1b74-7771-8a6f-0caba15cef8d', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'latest stock market news September 2026'}, 'id': 'chatcmpl-tool-b92fb8ea70ca96bd', 'type': 'tool_call'}], invalid_tool_calls=[])